# Lecture 8: From Language Model to AI Assistant

## Interactive Demo Notebook

In this notebook, we'll understand the complete journey:
1. **The Base Model Problem** - Why prediction isn't enough
2. **Supervised Fine-Tuning (SFT)** - Teaching to follow instructions
3. **RLHF Intuition** - Learning from human preferences
4. **The Full Pipeline** - Pre-training → SFT → RLHF
5. **Exploring Real Models** - Using transformers library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print("Ready to understand modern AI assistants! 🤖")

## Part 1: The Base Model Problem

A language model that predicts text well is NOT the same as a helpful assistant!

In [ ]:
# Simulate base model vs assistant behavior

prompt = "How do I make a chocolate cake?"

base_model_response = """
How do I make a chocolate cake?

I've been trying to make one for my daughter's birthday 
but they keep coming out dry. Any tips would be appreciated!

Posted by BakingMom2019 on RecipeForums.com
---
Re: How do I make a chocolate cake?

Try adding more eggs! Also make sure your...
"""

assistant_response = """
Here's a simple chocolate cake recipe:

**Ingredients:**
- 2 cups flour
- 2 cups sugar  
- 3/4 cup cocoa powder
- 2 eggs
- 1 cup milk
- 1/2 cup vegetable oil

**Steps:**
1. Preheat oven to 350°F (175°C)
2. Mix dry ingredients in a large bowl
3. Add wet ingredients and mix until smooth
4. Pour into greased 9x13 pan
5. Bake for 30-35 minutes

Let me know if you need frosting suggestions!
"""

print("="*60)
print("PROMPT:", prompt)
print("="*60)
print("\n📖 BASE MODEL (trained on internet text):")
print("-"*40)
print(base_model_response)
print("\n🤖 ASSISTANT MODEL (after SFT + RLHF):")
print("-"*40)
print(assistant_response)

print("\n💡 The base model completes web text!")
print("   The assistant actually ANSWERS the question!")

## Part 2: Supervised Fine-Tuning (SFT)

Teaching the model to follow instructions with examples

In [ ]:
# SFT Training Data Examples
sft_examples = [
    {
        "instruction": "What is the capital of France?",
        "response": "The capital of France is Paris."
    },
    {
        "instruction": "Write a haiku about programming.",
        "response": """Code flows like water
Bugs emerge from the shadows
Debug, compile, run"""
    },
    {
        "instruction": "Explain machine learning to a 10-year-old.",
        "response": """Imagine teaching a dog tricks. You show the dog what to do, 
give it treats when it does well, and it learns! Machine learning is 
like that, but for computers. We show computers lots of examples, 
and they learn to recognize patterns and make predictions."""
    }
]

print("SFT Training Data Examples:")
print("="*60)

for i, example in enumerate(sft_examples, 1):
    print(f"\nExample {i}:")
    print(f"  📥 Instruction: {example['instruction']}")
    print(f"  📤 Response: {example['response'][:80]}..." if len(example['response']) > 80 else f"  📤 Response: {example['response']}")

print("\n💡 SFT shows the model: 'When given X, respond like this'")
print("   It's like showing a new employee the expected format.")

In [ ]:
# Visualize the SFT process
fig, ax = plt.subplots(figsize=(12, 5))

# Boxes for the pipeline
stages = ['Base Model\n(Text Completer)', 'Instruction\nDataset', 'Fine-tuned\nModel']
x_positions = [1, 3, 5]
colors = ['#1e3a5f', '#e9c46a', '#2a9d8f']

for x, stage, color in zip(x_positions, stages, colors):
    rect = plt.Rectangle((x-0.6, 0.2), 1.2, 0.6, facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 0.5, stage, ha='center', va='center', fontsize=11, fontweight='bold', color='white')

# Arrows
ax.annotate('', xy=(2.4, 0.5), xytext=(1.6, 0.5),
            arrowprops=dict(arrowstyle='->', lw=2))
ax.annotate('', xy=(4.4, 0.5), xytext=(3.6, 0.5),
            arrowprops=dict(arrowstyle='->', lw=2))

# Labels
ax.text(2, 0.7, 'Train on', ha='center', fontsize=10)
ax.text(4, 0.7, 'Produces', ha='center', fontsize=10)

ax.set_xlim(0, 6)
ax.set_ylim(0, 1)
ax.axis('off')
ax.set_title('Supervised Fine-Tuning (SFT)', fontsize=14)
plt.show()

## Part 3: RLHF Intuition - Learning from Human Preferences

Which response is BETTER? Humans decide!

In [ ]:
# RLHF Preference Data Example
preference_example = {
    "prompt": "Explain quantum computing.",
    "response_A": """Quantum computing utilizes quantum mechanical phenomena 
such as superposition and entanglement to process information. Unlike classical 
bits which are either 0 or 1, quantum bits (qubits) can exist in multiple 
states simultaneously, enabling parallel computation.""",
    "response_B": """idk quantum stuff is weird lol. basically computers 
but with atoms or something. google it.""",
    "human_preference": "A"  # Humans prefer A
}

print("RLHF Preference Labeling:")
print("="*60)
print(f"\n📥 Prompt: {preference_example['prompt']}")
print("\n" + "-"*30 + " Response A " + "-"*30)
print(preference_example['response_A'])
print("\n" + "-"*30 + " Response B " + "-"*30)
print(preference_example['response_B'])
print("\n" + "="*60)
print(f"👤 Human Preference: {preference_example['human_preference']} (Response A is better)")

In [ ]:
# Visualize the RLHF pipeline
fig, ax = plt.subplots(figsize=(14, 6))

# Main flow
stages = [
    ('SFT Model', 0, 3, '#1e3a5f'),
    ('Generate\nResponses', 3, 3, '#3b82f6'),
    ('Human\nRanking', 6, 3, '#e9c46a'),
    ('Reward\nModel', 6, 1, '#8b5cf6'),
    ('RL\nOptimization', 9, 2, '#e85a4f'),
    ('Aligned\nModel', 12, 2, '#2a9d8f')
]

for name, x, y, color in stages:
    rect = plt.Rectangle((x-0.8, y-0.4), 1.6, 0.8, facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, name, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

# Arrows
arrows = [
    ((0.8, 3), (2.2, 3)),
    ((3.8, 3), (5.2, 3)),
    ((6, 2.6), (6, 1.4)),
    ((6.8, 1), (8.2, 2)),
    ((9.8, 2), (11.2, 2)),
]

for start, end in arrows:
    ax.annotate('', xy=end, xytext=start, arrowprops=dict(arrowstyle='->', lw=2))

ax.set_xlim(-1, 14)
ax.set_ylim(0, 4.5)
ax.axis('off')
ax.set_title('RLHF Pipeline: Learning from Human Preferences', fontsize=14)
plt.show()

print("💡 RLHF teaches the model WHAT humans consider 'good' responses")
print("   The reward model learns to predict human preferences")
print("   RL optimizes the model to get higher reward")

## Part 4: The Complete Pipeline

In [ ]:
# Visualize the complete journey
fig, ax = plt.subplots(figsize=(14, 4))

pipeline = [
    ('Internet\nText\n(TB)', 0, '#6b7280', 'Data'),
    ('Pre-training\n(Next Token)', 2, '#1e3a5f', 'Learn Language'),
    ('Base\nModel', 4, '#3b82f6', 'Text Completer'),
    ('SFT\nData', 5.5, '#6b7280', '~100K examples'),
    ('SFT', 6.5, '#e9c46a', 'Learn Instructions'),
    ('Instruct\nModel', 8, '#f59e0b', 'Follows Commands'),
    ('Human\nFeedback', 9.5, '#6b7280', 'Preferences'),
    ('RLHF', 10.5, '#e85a4f', 'Learn Values'),
    ('AI\nAssistant', 12, '#2a9d8f', 'ChatGPT/Claude')
]

for name, x, color, label in pipeline:
    # Main box
    rect = plt.Rectangle((x-0.6, 0.2), 1.2, 0.8, facecolor=color, edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, 0.6, name, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(x, -0.1, label, ha='center', fontsize=8, style='italic')

# Connect with arrows
for i in range(len(pipeline) - 1):
    x1 = pipeline[i][1] + 0.6
    x2 = pipeline[i+1][1] - 0.6
    ax.annotate('', xy=(x2, 0.6), xytext=(x1, 0.6),
                arrowprops=dict(arrowstyle='->', lw=1.5))

ax.set_xlim(-1, 13)
ax.set_ylim(-0.5, 1.5)
ax.axis('off')
ax.set_title('From Raw Text to AI Assistant: The Complete Journey', fontsize=14)
plt.tight_layout()
plt.show()

print("\n🎯 Key stages:")
print("   1. Pre-training: Learn language from massive text (months, millions $)")
print("   2. SFT: Learn to follow instructions (~100K examples, days)")
print("   3. RLHF: Learn human values and preferences (ongoing)")

## Part 5: Exploring Real Models (Optional)

In [ ]:
# Try loading a small model (if transformers is installed)
try:
    from transformers import pipeline
    
    # Use a small model for text generation
    generator = pipeline('text-generation', model='gpt2', max_length=50)
    
    prompts = [
        "The meaning of life is",
        "Machine learning is",
        "The capital of France is"
    ]
    
    print("GPT-2 (Base Model) Completions:")
    print("="*60)
    for prompt in prompts:
        result = generator(prompt, num_return_sequences=1, do_sample=True)
        print(f"\n📝 {result[0]['generated_text']}")
    
    print("\n💡 Notice: GPT-2 completes text but doesn't 'answer' questions!")
    print("   That's because it's a base model without SFT/RLHF.")
    
except ImportError:
    print("⚠️ transformers not installed. Run: pip install transformers")
    print("   Skipping real model demo.")

In [ ]:
# Compare model sizes
models = {
    'GPT-2': 1.5e9,
    'GPT-3': 175e9,
    'GPT-4': 1.76e12,  # Estimated
    'LLaMA-7B': 7e9,
    'LLaMA-70B': 70e9,
    'Claude-3': 1e12,  # Estimated
}

plt.figure(figsize=(10, 5))
names = list(models.keys())
params = [v/1e9 for v in models.values()]  # Convert to billions

colors = plt.cm.viridis(np.linspace(0, 0.8, len(models)))
bars = plt.bar(names, params, color=colors, edgecolor='white', linewidth=2)
plt.yscale('log')
plt.ylabel('Parameters (Billions)', fontsize=12)
plt.title('LLM Size Comparison (Log Scale)', fontsize=14)
plt.xticks(rotation=15)

# Add value labels
for bar, p in zip(bars, params):
    if p >= 1000:
        label = f'{p/1000:.1f}T'
    else:
        label = f'{p:.0f}B'
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.2,
             label, ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("💡 Larger models generally have more capabilities")
print("   But training and running them requires massive resources!")

## AI Ethics Considerations

In [ ]:
# Visualize key ethical considerations
concerns = {
    'Bias & Fairness': 'Models can reflect biases in training data',
    'Misinformation': 'Models can generate convincing but false content',
    'Privacy': 'Models may memorize sensitive training data',
    'Job Displacement': 'Automation may affect certain jobs',
    'Environmental Impact': 'Large models require significant energy',
    'Safety': 'Ensuring AI systems remain beneficial'
}

print("🚨 Key AI Ethics Considerations:")
print("="*60)
for concern, description in concerns.items():
    print(f"\n  ⚠️ {concern}")
    print(f"     {description}")

print("\n" + "="*60)
print("💡 As AI practitioners, we have a responsibility to:")
print("   - Consider potential harms")
print("   - Build with diverse perspectives")
print("   - Be transparent about limitations")
print("   - Prioritize safety and alignment")

## 🎯 Exercises

In [ ]:
# Exercise 1: Create 5 instruction-response pairs for SFT
# Think about what makes a good instruction dataset



In [ ]:
# Exercise 2: Think of a prompt and write two responses - one better than the other
# Explain why one is preferred (this is what human raters do!)



## Summary

| Stage | What Happens | Result |
|-------|--------------|--------|
| Pre-training | Learn language from internet | Text completer |
| SFT | Train on instruction pairs | Follows commands |
| RLHF | Learn from human preferences | Aligned assistant |

**The key insight:** Modern AI assistants are NOT just bigger language models - they're carefully aligned through multiple stages of training to be helpful, harmless, and honest.